# Google Cloud ADK 2.x Codelab: Secure Enterprise RAG in 10 Minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/enriquekalven/adk-ge-datastore-connector/blob/main/codelab.ipynb)
[![Google Cloud ADK](https://img.shields.io/badge/Google_Cloud-ADK_2.x-4285F4?logo=googlecloud&logoColor=white)](https://github.com/google/adk-python)
[![Gemini Enterprise](https://img.shields.io/badge/Gemini-Enterprise_Datastores-8E75B5?logo=google&logoColor=white)](https://cloud.google.com/vertex-ai)

This interactive Google Colab notebook guides you through implementing and verifying **server-side Access Control List (ACL) enforcement** for custom ADK agents over Gemini Enterprise datastores (SharePoint, Jira, Drive, Slack, BigQuery).


## 1. Clone Repository & Install Dependencies
Clone the reference architecture and install required dependencies.

In [ ]:
# Clone repository and install dependencies
import os, sys
if not os.path.exists("adk-ge-datastore-connector"):
    !git clone https://github.com/enriquekalven/adk-ge-datastore-connector.git
    %cd adk-ge-datastore-connector
else:
    %cd adk-ge-datastore-connector
!pip install -q -r requirements.txt pytest


## 2. Minimal ADK Tool Definition with 3-Legged OAuth (3LO)
Define a custom ADK search tool that extracts calling user session tokens dynamically from .

In [ ]:
from google.adk.agents import Agent
from google.adk.tools import ToolContext, tool
from tools.datastore_search import execute_datastore_query
from config import AuthMode

@tool
def search_enterprise_sharepoint(query: str, tool_context: ToolContext) -> str:
    """Searches corporate SharePoint documents enforcing calling user permissions."""
    return execute_datastore_query(
        query=query,
        tool_context=tool_context,
        engine_id="sharepoint-engine",
        auth_name="sharepoint_oauth",
        auth_mode=AuthMode.USER_OAUTH,
        project_id="my-gcp-project",
        location="global"
    )

root_agent = Agent(
    model="gemini-2.0-flash",
    name="enterprise_sharepoint_assistant",
    tools=[search_enterprise_sharepoint],
    instruction="Answer employee queries using search_enterprise_sharepoint. Always cite source links."
)
print("✅ ADK Root Agent successfully initialized with search_enterprise_sharepoint tool!")


## 3. Run the 10-Minute Codelab Demo
Demonstrates Alice (HR Manager), Bob (Junior Dev), Fail-Closed security verification, and BigQuery structured search.

In [ ]:
!python3 quickstart_demo.py


## 4. Run Preflight Diagnostic Sweep ()
Run the FDE diagnostic utility to validate endpoints and IAM roles.

In [ ]:
!python3 -m tools.doctor --json


## 5. Execute Full 39-Test Verification Suite
Verify the complete suite of 2LO, 3LO, Axis A, Axis B, and scale tests.

In [ ]:
!pytest -v
